# Lesson 07 - Corak Reka Bentuk Perancangan

Notebook ini menunjukkan **Corak Reka Bentuk Perancangan** untuk ejen AI menggunakan Rangka Kerja Ejen Microsoft.
Anda akan belajar bagaimana untuk memecahkan permintaan perjalanan yang kompleks kepada subtugas yang tersusun, memberikannya kepada ejen pakar,
dan melaksanakan pelan yang dihasilkan — semua menggunakan output tersusun yang dikuasakan oleh model Pydantic.


## Persediaan


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Penguraian Tugas

Penguraian tugas adalah teras kepada corak reka bentuk perancangan. Daripada meminta satu agen tunggal untuk mengendalikan permintaan yang kompleks dari awal hingga akhir, kami memecahkan masalah kepada **subtugas** yang lebih kecil dan jelas. Setiap subtugas diberikan kepada agen pakar (contohnya, penerbangan, hotel, aktiviti) dengan keutamaan dan susunan kebergantungan yang jelas.

Pendekatan ini memberikan beberapa manfaat:
- **Kejelasan**: setiap subtugas mempunyai tanggungjawab tunggal.
- **Selari**: subtugas bebas boleh dijalankan serentak.
- **Kebolehpercayaan**: kegagalan diasingkan kepada subtugas individu.
- **Penjejakan bajet**: kos dianggarkan bagi setiap subtugas dan dijumlahkan.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## Mewujudkan Ejen Perancangan dengan Keluaran Berstruktur

Ejen perancangan bertindak sebagai **penyelia meja hadapan**. Berdasarkan permintaan perjalanan peringkat tinggi, ia menghasilkan `TravelPlan` berstruktur — memecahkan permintaan kepada tugasan subtugas, menetapkan keutamaan, dan mengenal pasti kebergantungan supaya seorang concierge atau lapisan pelaksanaan dapat melaksanakan kerja tersebut.


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## Melaksanakan Pelan dengan Alat Pakar

Setelah ejen kaunter hadapan menghasilkan pelan yang terstruktur, **ejen concierge** melaksanakannya.
Setiap alat pakar mengendalikan satu kategori subtugas (penerbangan, hotel, aktiviti). Concierge
melalui subtugas pelan mengikut urutan kebergantungan dan menghantar setiap satu ke
alat yang sesuai.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## Ringkasan

Dalam pelajaran ini anda telah mempelajari **Reka Bentuk Corak Perancangan** bagi ejen AI:

1. **Penguraian Tugasan** — Ejen perancang meja depan memecahkan permintaan perjalanan yang kompleks kepada tugasan subtugas berstruktur menggunakan model Pydantic, memberikan setiap satu kepada ejen pakar dengan keutamaan dan kebergantungan.
2. **Output Berstruktur** — Dengan menghantar `response_format` ejen mengembalikan objek `TravelPlan` yang disahkan dan bukannya teks bebas, menjadikan pemprosesan seterusnya lebih boleh dipercayai.
3. **Pelaksanaan Pelan** — Ejen konsièrge mengulangi subtugas menggunakan alat pakar (`book_flight`, `reserve_hotel`, `book_activity`) untuk melaksanakan pelan dan melaporkan keputusan.

Corak ini memisahkan *apa yang perlu dilakukan* (perancangan) daripada *bagaimana untuk melakukannya* (pelaksanaan), menjadikan ejen lebih modular, mudah diuji, dan lebih mudah untuk dikembangkan.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk memastikan ketepatan, sila ambil perhatian bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan profesional oleh manusia adalah disyorkan. Kami tidak bertanggungjawab atas sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
